<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-08-agents-and-adk/lesson-8.3-agent-engine/practice/GCP_Capstone_8.3_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 8.3 — Agent Engine: Managed Deployment

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup: install, authenticate, configure

Run this section once. It installs the Agent Engine + ADK SDK, authenticates with Application Default Credentials (no API keys), sets the Vertex environment variables, and builds the DocuMind agent that the deployment exercises below reuse. Exercise 4 revisits the Memory Bank wiring in depth.

In [ ]:
%%bash
pip install -q 'google-cloud-aiplatform[agent_engines,adk]>=1.112'

In [ ]:
import os

# Application Default Credentials on Colab (no API keys). No-op off Colab.
try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass

PROJECT_ID = 'documind-ai-YOUR-ID'
LOCATION = 'us-central1'          # asia-south1 for India production
USD_INR = 85

os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
os.environ['GOOGLE_CLOUD_LOCATION'] = LOCATION
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'TRUE'
print('Setup complete')

In [ ]:
# Build the DocuMind agent (from Lesson 8.1, wired for Memory Bank).
# The deploy/query exercises reuse this `root_agent`; Exercise 4 explains the wiring.
from google.adk.agents import Agent
from google.adk.tools import ToolContext
from google.adk.tools.preload_memory_tool import PreloadMemoryTool
from google.adk.agents.callback_context import CallbackContext

# Memory write callback: persists each finished session into Memory Bank
async def save_memory(callback_context: CallbackContext):
    await callback_context.add_session_to_memory()
    return None

def search_documents(query: str, tool_context: ToolContext) -> dict:
    """Search documents.
    Args:
        query: Search query.
    """
    h = tool_context.state.get('search_history', [])
    h.append(query)
    tool_context.state['search_history'] = h
    return {'results': [{'id': 'D-01', 'title': 'Q1 Report'}]}

def summarize_document(document_id: str, summary_type: str) -> dict:
    """Summarize a document.
    Args:
        document_id: Document ID.
        summary_type: brief/detailed/executive.
    """
    return {'summary': f'Summary of {document_id}'}

root_agent = Agent(
    name='documind',
    model='gemini-3.6-flash',
    instruction='You are DocuMind AI. Use past context when relevant.',
    tools=[PreloadMemoryTool(), search_documents, summarize_document],
    after_agent_callback=save_memory,
)
print('DocuMind agent created')

## Exercise 1: Deploy to Agent Engine

**Difficulty:** Easy

`adk deploy agent_engine` with your Lesson 8.1 agent.

1. Create a `vertexai.Client` for your project/location.
2. Wrap `root_agent` in an `AdkApp` (tracing on for observability).
3. Call `client.agent_engines.create(...)` with a staging bucket and instance limits.

**Expected behaviour:** Resource name returned. Agent deployed.

In [ ]:
import vertexai
from vertexai.agent_engines import AdkApp

client = vertexai.Client(
    project=os.environ['GOOGLE_CLOUD_PROJECT'],
    location=os.environ['GOOGLE_CLOUD_LOCATION'])

adk_app = AdkApp(agent=root_agent, enable_tracing=True)

# Deploy (takes ~10 minutes; requires a GCP project with billing).
# remote = client.agent_engines.create(
#     agent=adk_app,
#     config={
#         'display_name': 'DocuMind Production',
#         'requirements': ['google-cloud-aiplatform[agent_engines,adk]'],
#         'staging_bucket': 'gs://documind-staging',
#         'min_instances': 1,
#         'max_instances': 10,
#     }
# )
# print(f'Deployed: {remote.resource_name}')
print('Uncomment to deploy (requires GCP project with billing)')

In [ ]:
%%bash
# CLI alternative to the programmatic create() above.
# Point it at the folder that exports `root_agent`, then deploy:
# adk deploy agent_engine \
#   --project "$GOOGLE_CLOUD_PROJECT" \
#   --region "$GOOGLE_CLOUD_LOCATION" \
#   --staging_bucket gs://documind-staging \
#   --display_name "DocuMind Production" \
#   ./documind_agent
echo 'Uncomment to deploy via the adk CLI'

## Exercise 2: Query Deployed Agent

**Difficulty:** Easy

`async_stream_query()` to send a message. Print response.

1. Create a session for a user on the deployed `remote` agent.
2. Stream a query with `async_stream_query(...)`.
3. Print each streamed event as it arrives.

**Expected behaviour:** Agent responds via streaming events.

In [ ]:
# Uncomment once the agent is deployed (Exercise 1 -> `remote`).
# session = await remote.async_create_session(user_id='student1')
# async for event in remote.async_stream_query(
#     user_id='student1',
#     session_id=session['id'],
#     message='Find documents about revenue'
# ):
#     print(event)
print('Deploy first (Exercise 1), then uncomment to query')

## Exercise 3: Sessions

**Difficulty:** Easy

Create session. Send 3 messages. List all sessions.

1. Create a session for `user_id='student1'`.
2. Stream three messages into the same `session_id` so they share one conversation.
3. List all of that user's sessions to confirm persistence.

**Expected behaviour:** 3 turns. Session persisted.

In [ ]:
# Uncomment once the agent is deployed (Exercise 1 -> `remote`).
# session = await remote.async_create_session(user_id='student1')
#
# for msg in ['Find revenue docs', 'Summarize D-01', 'What did I just ask about?']:
#     async for event in remote.async_stream_query(
#         user_id='student1', session_id=session['id'], message=msg):
#         print(event)
#
# # List every session for this user
# async for s in remote.async_list_sessions(user_id='student1'):
#     print('session:', s['id'])
print('Deploy first (Exercise 1), then uncomment to run the 3-turn session')

## Exercise 4: Wire Memory Bank

**Difficulty:** Medium

`PreloadMemoryTool` + `after_agent_callback`. Verify memories persist across sessions.

1. Add `PreloadMemoryTool()` to the agent so past memories are injected at the start of a run.
2. Register an `after_agent_callback` that calls `add_session_to_memory()` to write the finished session into Memory Bank.
3. Run a second session for the same user and confirm it recalls facts from the first.

**Expected behaviour:** Session 2 recalls facts from Session 1.

In [ ]:
# This is the Memory Bank wiring already applied to `root_agent` in Setup.
# The two moving parts:
#   - PreloadMemoryTool()  -> reads relevant memories in before the agent responds
#   - after_agent_callback -> writes the session out after the agent responds

async def save_memory(callback_context: CallbackContext):
    await callback_context.add_session_to_memory()
    return None

root_agent = Agent(
    name='documind',
    model='gemini-3.6-flash',
    instruction='You are DocuMind AI. Use past context when relevant.',
    tools=[PreloadMemoryTool(), search_documents, summarize_document],
    after_agent_callback=save_memory,
)
print('Memory Bank wired: PreloadMemoryTool (read) + save_memory callback (write)')

In [ ]:
# Verify persistence across sessions (uncomment once deployed).
# # Session 1: teach it a fact
# s1 = await remote.async_create_session(user_id='student1')
# async for e in remote.async_stream_query(
#     user_id='student1', session_id=s1['id'],
#     message='Remember: our fiscal year ends in March.'):
#     print(e)
# await remote.async_add_session_to_memory(user_id='student1', session_id=s1['id'])
#
# # Session 2: a brand-new session should recall it via PreloadMemoryTool
# s2 = await remote.async_create_session(user_id='student1')
# async for e in remote.async_stream_query(
#     user_id='student1', session_id=s2['id'],
#     message='When does our fiscal year end?'):
#     print(e)
print('Deploy first, then uncomment to verify Session 2 recalls Session 1')

## Exercise 5: Context Caching

**Difficulty:** Medium

Configure `ContextCacheConfig`. Compare token costs with/without caching.

1. Build a `ContextCacheConfig` (min tokens, TTL, refresh interval) and attach it to the `App`.
2. Model a large shared context reused across many queries.
3. Compare billed input tokens/cost with caching vs without.

**Expected behaviour:** Cached queries show ~90% fewer input tokens billed.

In [ ]:
from google.adk.apps.app import App
from google.adk.agents.context_cache_config import ContextCacheConfig

cache_config = ContextCacheConfig(
    min_tokens=2048,     # only cache prefixes this large or larger
    ttl_seconds=600,     # keep the cache warm for 10 minutes
    cache_intervals=5,   # refresh the cache every 5 invocations
)

app = App(
    name='documind',
    root_agent=root_agent,
    context_cache_config=cache_config,
)
print('App configured with context caching')

In [ ]:
# Token-cost comparison: a 50K-token shared context reused across 100 queries.
FLASH_INPUT_PER_1M = 1.50   # USD, gemini-3.6-flash standard input
USD_INR = 85

shared_context_tokens = 50_000
num_queries = 100

# Without caching: full context billed every call.
uncached_input = shared_context_tokens * num_queries
# With caching: first call pays full price, cache hits bill the prefix at ~10%.
cached_input = shared_context_tokens + shared_context_tokens * 0.10 * (num_queries - 1)

uncached_usd = uncached_input * FLASH_INPUT_PER_1M / 1e6
cached_usd = cached_input * FLASH_INPUT_PER_1M / 1e6
savings = 1 - cached_input / uncached_input

print(f'Without caching: {uncached_input:>10,} input tokens  ${uncached_usd:6.2f}  Rs.{uncached_usd*USD_INR:8.2f}')
print(f'With caching:    {cached_input:>10,.0f} input tokens  ${cached_usd:6.2f}  Rs.{cached_usd*USD_INR:8.2f}')
print(f'Input tokens billed reduced by {savings*100:.0f}%')

## Exercise 6: Context Compaction

**Difficulty:** Medium

`EventsCompactionConfig` with `interval=3`. Test with a 10-turn conversation.

1. Add `EventsCompactionConfig(compaction_interval=3, overlap_size=1)` to the `App` (alongside caching).
2. Walk a 10-turn conversation.
3. Show where compaction fires and older events collapse into a running summary.

**Expected behaviour:** Older events summarized. Token count reduced.

In [ ]:
from google.adk.apps.app import App, EventsCompactionConfig

app = App(
    name='documind',
    root_agent=root_agent,
    context_cache_config=cache_config,
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=3,   # summarize every 3 events
        overlap_size=1,          # keep 1 recent event verbatim for continuity
    ),
)
print('App configured with caching + compaction')

In [ ]:
# Illustrate where compaction fires over a 10-turn conversation.
# With compaction_interval=3, checkpoints land after turns 3, 6, 9.
for turn in range(1, 11):
    if turn % 3 == 0:
        print(f'Turn {turn:>2}: COMPACTION -> events 1..{turn} summarized (overlap=1 kept verbatim)')
    else:
        print(f'Turn {turn:>2}: appended')

## Exercise 7: HIPAA Checklist

**Difficulty:** Challenge

Document a complete HIPAA checklist for DocuMind. Execute BAA.

1. Enumerate the 8 controls (BAA, encryption, network, access, models, region, DLP, retention).
2. Print the checklist.
3. Add the India DPDP Act obligations alongside HIPAA.

**Expected behaviour:** 8-item checklist documented.

In [ ]:
hipaa_checklist = {
    '1_baa': 'Execute BAA via Cloud Console > Privacy & Security > Legal',
    '2_encryption': 'AES-256 at rest (default) + CMEK + TLS 1.2+ in transit',
    '3_network': 'VPC Service Controls perimeter around Vertex AI resources',
    '4_access': 'IAM least-privilege + Cloud Audit Logs enabled',
    '5_models': 'Use only BAA-covered models (verify each model)',
    '6_region': 'Regional endpoints only (us-central1). Never global endpoint for PHI.',
    '7_dlp': 'Cloud DLP scanning on data flows for PHI detection',
    '8_retention': 'Configure zero data retention at project level',
}

print('HIPAA Compliance Checklist for DocuMind:')
for k, v in hipaa_checklist.items():
    print(f'  [{k}] {v}')

print('\nIndia DPDP Act:')
print('  - Consent before processing personal data')
print('  - Data Principal rights (access/correction/erasure, 7-day window)')
print('  - 72-hour breach notification')
print('  - Use asia-south1 for Indian users')
print('  - Penalties: up to Rs.250 crore per violation')

## Exercise 8: Full Production Deploy

**Difficulty:** Challenge

Deploy multi-agent DocuMind with Memory Bank + caching + monitoring.

1. Confirm the agent (Memory Bank) and `App` (caching + compaction) are configured.
2. Wrap in an `AdkApp` with tracing enabled for observability.
3. Deploy with instance limits; monitor via Cloud Trace / Agent Engine metrics.

**Expected behaviour:** Full production system with observability.

In [ ]:
# Consolidates Exercises 1-6: memory-wired agent + caching/compaction App + traced deploy.
assert root_agent is not None, 'Build root_agent in Setup / Exercise 4 first'
assert app is not None, 'Build the App in Exercises 5-6 first'

adk_app = AdkApp(agent=root_agent, enable_tracing=True)   # tracing -> Cloud Trace

# remote = client.agent_engines.create(
#     agent=adk_app,
#     config={
#         'display_name': 'DocuMind Production',
#         'requirements': ['google-cloud-aiplatform[agent_engines,adk]'],
#         'staging_bucket': 'gs://documind-staging',
#         'min_instances': 1,
#         'max_instances': 10,
#     }
# )
# print(f'Deployed: {remote.resource_name}')
#
# Observability once live:
#   - enable_tracing=True streams spans to Cloud Trace
#   - Agent Engine surfaces request/latency/error metrics in Cloud Monitoring
#   - after_agent_callback keeps Memory Bank populated across sessions
print('Uncomment to deploy the full production DocuMind (Memory Bank + caching + tracing)')